In [2]:
import tqdm
import warnings
import quantstats as qs
from utils.evaluate import *
from utils.data_bridge import *
from method import *
warnings.filterwarnings('ignore')
qs.extend_pandas()

In [3]:
# 定義開盤漲跌幅和日內漲跌幅
handler = Handler('data')
handler.start_date = '2019-01-01'

adj_close = handler['Close'] * handler['Adjust_Factor']
adj_open = handler['Open'] * handler['Adjust_Factor']

# 隔夜
exp_overnight_ret = adj_open.shift(-2) / adj_close.shift(-1) - 1
# 日內
exp_intrday_ret = adj_close.shift(-1) / adj_open.shift(-1) - 1
# 波段
exp_day_ret = adj_open.shift(-2) / adj_open.shift(-1) - 1

In [4]:
# 濾網
Close = handler['Close']
Values = handler['Value_Dollars'].shift(1)

# 濾網
上市超過5天 = handler['Close'].notna().astype(int).rolling(5).sum()==5
成交額濾網 = handler['Value_Dollars'] >= 20_000_000
開盤非漲停 = ~((handler['Limit_Up_or_Down_in_Opening_Fg'] == 'Y') & (adj_open > adj_close.shift(1)))
買賣當沖 = handler['Suspension_of_buy_After_Day_Trading_Fg'] != 'Y'
非漲停 = handler['Limit_Up_or_Down'] != '+'

濾網 = 上市超過5天 & 成交額濾網 & 開盤非漲停.shift(-1)
隔夜濾網 = 成交額濾網 & 非漲停.shift(-1)
日內濾網 = 成交額濾網 & 買賣當沖.shift(-1)

In [ ]:
# 單因子分析
expr = "ts_mean(Custodied_Greater_Than_1000_Lots_Pct, 21) - ts_mean(Custodied_Under_400_Lots_Pct, 21)"
expr = "cs_zscore(interaction_div(Cash_Flow_Ratio_TTM, Debt_To_Equity_Ratio_TTM))"
factor = eval(expr, funcs_methods, handler)

# 設定濾網
factor = factor.where(日內濾網, np.nan)
exp_ret = exp_intrday_ret.where(日內濾網, np.nan)
ret = factors_analyze(factor, exp_ret, one_side=False, rank_range_n=20, quantile_metric='mean')

1. 分組統計指標表（bps） — Factor (方向: +)...


,計數,比例(%),平均(bps),中位數(bps),標準差(bps)
X 範: Factor,,,,,
"(-99.869, -52.191]",50894,4.92,-33.3243,-46.7290,274.9631
"(-52.191, -41.002]",51742,5.00,-32.6051,-45.1467,269.1997
"(-41.002, -32.554]",51664,5.00,-30.2931,-42.0168,273.4785
"(-32.554, -24.895]",51817,5.01,-30.5252,-41.9287,269.6458
"(-24.895, -18.606]",51720,5.00,-28.9642,-38.8853,262.3009
"(-18.606, -12.723]",51582,4.99,-30.8490,-39.6825,257.8168
"(-12.723, -6.938]",51654,5.00,-28.5572,-38.5480,263.9246
"(-6.938, -1.499]",51811,5.01,-28.8449,-36.3636,258.1408
"(-1.499, 4.230]",51573,4.99,-28.5498,-37.5940,258.0963



2. 繪製分組 mean 比較圖...


3. 繪製累積收益曲線圖...


統計指標:


,CAGR(%),Sharpe,MDD(%),單利MDD(%),IC,ICIR,樣本勝率(%),周勝率(%),月勝率(%),年勝率(%),盈虧比,總賺賠比,預期報酬(bps),樣本數
Factor,23.18,3.78,-2.67,-1.9,0.0612,0.4183,60.2,68.29,87.06,100.0,1.24,1.87,8.27,1716
